# Modeling

Trains and evaluates three classifiers on the prepared dataset, all on the same train/test split for direct comparability:

1. **TF-IDF + LogisticRegression** — lexical-feature baseline. Also serves as a diagnostic probe: if it matches the bigger models, the task is too lexically separable and the harder negatives aren't doing enough work.
2. **Sentence-Transformer (MiniLM) + LogisticRegression** — frozen semantic embeddings with a linear head. The "smart features, simple model" middle path.
3. **DistilBERT** — end-to-end fine-tune with weighted cross-entropy.

All three go through a shared evaluation function reporting per-class metrics, per-bucket recall (the headline diagnostic), PR curves, AP scores, and operating points at fixed FPR.

**Runtime**: ~10–15 minutes on Colab free tier with T4 GPU. Set runtime to GPU before running the DistilBERT cell.

## Environment setup

In [ ]:
!pip install -q datasets sentence-transformers scikit-learn pandas pyarrow transformers accelerate matplotlib

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=True)
DATA_DIR = Path("/content/drive/MyDrive/biology_refusal/data")
OUT_DIR  = Path("/content/drive/MyDrive/biology_refusal/models")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUT_DIR:  {OUT_DIR}")

## Load splits

In [ ]:
train_df = pd.read_parquet(DATA_DIR / "train.parquet")
test_df  = pd.read_parquet(DATA_DIR / "test.parquet")

print(f"Train: {len(train_df)}  (pos={train_df['label'].sum()}, neg={(train_df['label']==0).sum()})")
print(f"Test:  {len(test_df)}   (pos={test_df['label'].sum()}, neg={(test_df['label']==0).sum()})")
print(f"\nTrain source breakdown:\n{train_df['source'].value_counts()}")
print(f"\nTest source breakdown:\n{test_df['source'].value_counts()}")

X_train_text = train_df["text"].tolist()
y_train      = train_df["label"].to_numpy()
X_test_text  = test_df["text"].tolist()
y_test       = test_df["label"].to_numpy()

## Shared evaluation function

In [ ]:
# All three models go through this. Reports the headline metrics plus
# per-bucket recall on negatives — the most important diagnostic.

def evaluate(name, y_true, y_pred, y_score, test_df):
    """
    Print full eval report for one model.
    y_pred: hard predictions at threshold 0.5
    y_score: probability scores for the positive class (for PR/AP)
    """
    print(f"\n{'='*70}")
    print(f"  {name}")
    print(f"{'='*70}")

    # Per-class metrics
    print("\nPer-class report:")
    print(classification_report(
        y_true, y_pred,
        target_names=["don't refuse (0)", "refuse (1)"],
        digits=3,
    ))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    print(f"Confusion matrix:")
    print(f"           pred 0  pred 1")
    print(f"  true 0:  {cm[0,0]:6d}  {cm[0,1]:6d}")
    print(f"  true 1:  {cm[1,0]:6d}  {cm[1,1]:6d}")

    # Threshold-free metrics
    ap = average_precision_score(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)
    print(f"\nAP (area under PR curve): {ap:.4f}")
    print(f"ROC-AUC:                  {auc:.4f}")

    # Per-bucket recall on NEGATIVES — the headline diagnostic.
    # Reveals whether the model learned surface features (low recall on
    # mmlu_bio = "biology = refuse" failure mode).
    print("\nPer-bucket recall on negatives:")
    print("  (fraction of bucket correctly identified as 'don't refuse')")
    test_df_eval = test_df.copy()
    test_df_eval["pred"] = y_pred
    for bucket in sorted(test_df_eval["source"].unique()):
        sub = test_df_eval[test_df_eval["source"] == bucket]
        if sub["label"].iloc[0] == 1:
            continue  # skip positives
        recall = (sub["pred"] == 0).mean()
        print(f"  {bucket:25s}  n={len(sub):4d}  recall={recall:.3f}")

    # Per-source recall on POSITIVES (original vs rephrased)
    print("\nPer-source recall on positives:")
    print("  (fraction correctly identified as 'refuse')")
    for source in sorted(test_df_eval[test_df_eval["label"] == 1]["source"].unique()):
        sub = test_df_eval[test_df_eval["source"] == source]
        recall = (sub["pred"] == 1).mean()
        print(f"  {source:25s}  n={len(sub):4d}  recall={recall:.3f}")

    # Operating points at fixed FPR — what production-ish thresholds look like
    print("\nOperating points (threshold tuned for fixed FPR):")
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_score)
    # Compute FPR at each threshold
    fpr_at_thr = []
    for thr in np.sort(np.unique(y_score))[::-1][:200]:
        preds = (y_score >= thr).astype(int)
        tn = ((preds == 0) & (y_true == 0)).sum()
        fp = ((preds == 1) & (y_true == 0)).sum()
        if tn + fp == 0:
            continue
        fpr = fp / (fp + tn)
        tp = ((preds == 1) & (y_true == 1)).sum()
        fn = ((preds == 0) & (y_true == 1)).sum()
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr_at_thr.append((thr, fpr, rec))

    for target_fpr in [0.01, 0.05, 0.10]:
        # Find threshold with highest recall at FPR <= target
        valid = [(thr, fpr, rec) for thr, fpr, rec in fpr_at_thr if fpr <= target_fpr]
        if valid:
            best = max(valid, key=lambda x: x[2])
            print(f"  FPR <= {target_fpr:.2f}:  threshold={best[0]:.3f}, "
                  f"actual FPR={best[1]:.3f}, recall={best[2]:.3f}")
        else:
            print(f"  FPR <= {target_fpr:.2f}:  no threshold achieves this")

    return {"name": name, "ap": ap, "auc": auc, "y_score": y_score, "y_pred": y_pred}

## Model 1 — TF-IDF + LogisticRegression

In [ ]:
# Lexical-feature baseline. DIAGNOSTIC PURPOSE: if this gets >95% F1 on hard
# buckets (mmlu_bio, dual_use), the task is too separable by lexical features
# and our hard negatives aren't doing enough work. If it underperforms there
# but the embedding/transformer models do well, that's evidence the harder
# negatives are creating real semantic signal.

print("Training Model 1: TF-IDF + LogisticRegression...")

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    max_features=50000,
)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf  = tfidf.transform(X_test_text)
print(f"  TF-IDF feature dim: {X_train_tfidf.shape[1]}")

# class_weight='balanced' handles the 3.5:1 negative skew.
# LogisticRegressionCV picks C via 5-fold CV on training set.
tfidf_lr = LogisticRegressionCV(
    Cs=10, cv=5, max_iter=2000, class_weight="balanced",
    scoring="f1", random_state=SEED, n_jobs=-1,
)
tfidf_lr.fit(X_train_tfidf, y_train)
print(f"  Best C: {tfidf_lr.C_[0]:.4f}")

tfidf_pred  = tfidf_lr.predict(X_test_tfidf)
tfidf_score = tfidf_lr.predict_proba(X_test_tfidf)[:, 1]
tfidf_result = evaluate("Model 1: TF-IDF + LogReg", y_test, tfidf_pred, tfidf_score, test_df)

## Model 2 — Sentence-transformer embeddings + LogReg

In [ ]:
# Semantic-feature workhorse. Frozen MiniLM encoder + linear classifier on top.
# Embeddings cached for speed; CV used for the regularization strength.

print("\nTraining Model 2: Sentence-transformer + LogReg...")
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-MiniLM-L6-v2")

# Encode train and test (cached to disk to avoid recomputation on re-runs).
EMB_DIR = OUT_DIR / "embeddings"
EMB_DIR.mkdir(exist_ok=True)
train_emb_path = EMB_DIR / "train_minilm.npy"
test_emb_path  = EMB_DIR / "test_minilm.npy"

if train_emb_path.exists() and test_emb_path.exists():
    print("  Loading cached embeddings...")
    X_train_emb = np.load(train_emb_path)
    X_test_emb  = np.load(test_emb_path)
else:
    print("  Encoding train...")
    X_train_emb = encoder.encode(
        X_train_text, batch_size=64, show_progress_bar=True, convert_to_numpy=True,
    )
    print("  Encoding test...")
    X_test_emb  = encoder.encode(
        X_test_text, batch_size=64, show_progress_bar=True, convert_to_numpy=True,
    )
    np.save(train_emb_path, X_train_emb)
    np.save(test_emb_path, X_test_emb)
    print(f"  Cached embeddings to {EMB_DIR}/")

print(f"  Embedding dim: {X_train_emb.shape[1]}")

st_lr = LogisticRegressionCV(
    Cs=10, cv=5, max_iter=2000, class_weight="balanced",
    scoring="f1", random_state=SEED, n_jobs=-1,
)
st_lr.fit(X_train_emb, y_train)
print(f"  Best C: {st_lr.C_[0]:.4f}")

st_pred  = st_lr.predict(X_test_emb)
st_score = st_lr.predict_proba(X_test_emb)[:, 1]
st_result = evaluate(
    "Model 2: Sentence-Transformer (MiniLM) + LogReg",
    y_test, st_pred, st_score, test_df,
)

## Model 3 — DistilBERT fine-tune

In [ ]:
# End-to-end fine-tune with strong regularization to handle the ~6k training
# examples. Weight decay 0.01, dropout (default 0.1), 3 epochs, lr 2e-5,
# weighted loss for the 3.5:1 negative skew.
#
# Requires a GPU runtime. On Colab T4 this takes ~5-8 minutes.

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nTraining Model 3: DistilBERT fine-tune on {device}...")

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.encodings = tokenizer(
            texts, truncation=True, padding="max_length",
            max_length=max_length, return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         self.labels[idx],
        }

train_ds = TextDataset(X_train_text, y_train, tokenizer)
test_ds  = TextDataset(X_test_text,  y_test,  tokenizer)

# Class weights for weighted cross-entropy.
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
class_weights = torch.tensor(
    [len(y_train) / (2 * n_neg), len(y_train) / (2 * n_pos)],
    dtype=torch.float,
).to(device)
print(f"  Class weights: neg={class_weights[0]:.3f}, pos={class_weights[1]:.3f}")

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir=str(OUT_DIR / "distilbert"),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    save_strategy="no",       # avoid filling Colab disk
    eval_strategy="no",       # we evaluate manually below
    seed=SEED,
    report_to="none",
    fp16=(device == "cuda"),
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
)
trainer.train()

## DistilBERT evaluation

In [ ]:
print("\nEvaluating DistilBERT...")
model.eval()
all_logits = []
with torch.no_grad():
    # Process in batches manually to keep memory low.
    batch_size = 32
    for i in range(0, len(test_ds), batch_size):
        batch = [test_ds[j] for j in range(i, min(i + batch_size, len(test_ds)))]
        input_ids      = torch.stack([b["input_ids"]      for b in batch]).to(device)
        attention_mask = torch.stack([b["attention_mask"] for b in batch]).to(device)
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        all_logits.append(logits.cpu().numpy())

all_logits = np.concatenate(all_logits, axis=0)
# Softmax to probabilities.
exp = np.exp(all_logits - all_logits.max(axis=1, keepdims=True))
probs = exp / exp.sum(axis=1, keepdims=True)
db_score = probs[:, 1]
db_pred  = (db_score >= 0.5).astype(int)

db_result = evaluate("Model 3: DistilBERT fine-tune", y_test, db_pred, db_score, test_df)

## Side-by-side comparison + PR curves

In [ ]:
print("\n" + "=" * 70)
print("  HEADLINE COMPARISON")
print("=" * 70)

results = [tfidf_result, st_result, db_result]
summary = pd.DataFrame([
    {
        "model": r["name"],
        "F1 (pos)":  f1_score(y_test, r["y_pred"], pos_label=1),
        "F1 (neg)":  f1_score(y_test, r["y_pred"], pos_label=0),
        "AP":        r["ap"],
        "ROC-AUC":   r["auc"],
    }
    for r in results
])
print(summary.to_string(index=False))

# PR curves overlay
fig, ax = plt.subplots(figsize=(7, 5))
for r in results:
    p, rec, _ = precision_recall_curve(y_test, r["y_score"])
    ax.plot(rec, p, label=f"{r['name'].split(':')[0]} (AP={r['ap']:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall curves (positive = refuse)")
ax.legend(loc="lower left", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "pr_curves.png", dpi=120)
plt.show()

# Per-bucket recall comparison — the most important table for the write-up.
print("\n" + "=" * 70)
print("  PER-BUCKET RECALL ON NEGATIVES (the headline diagnostic)")
print("=" * 70)

bucket_table = []
for r in results:
    row = {"model": r["name"].split(":")[0]}
    test_df_eval = test_df.copy()
    test_df_eval["pred"] = r["y_pred"]
    for bucket in sorted(test_df_eval[test_df_eval["label"] == 0]["source"].unique()):
        sub = test_df_eval[test_df_eval["source"] == bucket]
        row[bucket] = (sub["pred"] == 0).mean()
    bucket_table.append(row)
print(pd.DataFrame(bucket_table).to_string(index=False))

## Error analysis — pull misclassifications for inspection

In [ ]:
# For each model, pull a sample of false positives and false negatives.
# Save to CSV for manual review (the task's "calling out failure modes" ask).

print("\nWriting error analysis files...")
err_dir = OUT_DIR / "errors"
err_dir.mkdir(exist_ok=True)

for r in results:
    name = r["name"].split(":")[0].lower().replace(" ", "_")
    test_df_err = test_df.copy()
    test_df_err["pred"]  = r["y_pred"]
    test_df_err["score"] = r["y_score"]

    # False positives: predicted refuse, actually don't refuse
    # — sorted by score descending (most confident wrong predictions first)
    fp = test_df_err[(test_df_err["label"] == 0) & (test_df_err["pred"] == 1)]
    fp = fp.sort_values("score", ascending=False)

    # False negatives: predicted don't refuse, actually refuse
    # — sorted by score ascending (most confident wrong predictions first)
    fn = test_df_err[(test_df_err["label"] == 1) & (test_df_err["pred"] == 0)]
    fn = fn.sort_values("score", ascending=True)

    fp.to_csv(err_dir / f"{name}_false_positives.csv", index=False)
    fn.to_csv(err_dir / f"{name}_false_negatives.csv", index=False)
    print(f"  {name}: {len(fp)} FPs, {len(fn)} FNs  →  {err_dir}/")

# Save full predictions for reproducibility.
preds_df = test_df.copy()
for r in results:
    name = r["name"].split(":")[0].lower().replace(" ", "_")
    preds_df[f"{name}_pred"]  = r["y_pred"]
    preds_df[f"{name}_score"] = r["y_score"]
preds_df.to_parquet(OUT_DIR / "all_predictions.parquet", index=False)
print(f"\nAll predictions saved to {OUT_DIR}/all_predictions.parquet")

## Cluster-of-origin recall check

In [ ]:
# We flagged that the test set is dominated by SMALL WMDP clusters because of
# the cumulative-size greedy allocation. If small-cluster test positives have
# systematically lower recall than would be expected, that confirms our test
# set is a "harder than average" slice — important for honest write-up framing.
#
# We don't have cluster IDs in the saved parquet (intentionally dropped), so
# this is a placeholder noting what to check if you re-run the data pipeline
# with cluster IDs preserved. Skip for now or re-run pipeline with cluster
# info if you want this analysis.

print("\nNote: cluster-of-origin recall analysis requires re-running the data")
print("pipeline with cluster IDs preserved in the saved parquet. Skipping.")